In [51]:
import pandas as pd 
import sklearn as sk
import os
import pandas as pd
import numpy as np 
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
import scanpy as sc
import anndata as ad
import bbknn
from sklearn.decomposition import PCA
import numpy as np
import harmonypy as hm
GENE_PANEL = ["ATOH1","DLL1","DLL4","GFI1","AREG","HES1","HES5","JAG2","NOTCH1","NOTCH2","NOTCH3",
              "OLFM4","LEF1","APCDD1","WNT6","NEUROG3","NEUROD1","KRT20","NEURL1","LGR5"]

In [68]:
data_path='/mnt/cold2/snaketree/prj/PPH/local/share/data/saver_mat'
train_id=['CRC0322LMX','CRC0322LMO']
tratt_cercato=['CTX1w_1','NT1w_1','CTX72h_1','NT72h_1']
train=pd.DataFrame()
for file in os.listdir(data_path):
    sample_name=str.split(file,sep='_')[4]
    trattamento=str.split(file,sep='_')[5]+'_'+str.split(file,sep='_')[6]
    if sample_name in train_id and trattamento in tratt_cercato:
        print(sample_name)
        data=pd.read_csv(os.path.join(data_path,file),header=0,index_col=0)
        data=data.T
        data['sample']=sample_name
        data['cell_id']=data.index
        data['trattamento']=trattamento
        data.reset_index(drop=True,inplace=True)
        train=pd.concat([train,data])

CRC0322LMO
CRC0322LMO
CRC0322LMX
CRC0322LMX


In [69]:
meta_cols = ["cell_id", "sample", "trattamento"]
expr_cols = [c for c in train.columns if c not in meta_cols]

# sostituisco i NaN con 0 solo nelle colonne di espressione
train[expr_cols] = train[expr_cols].fillna(0)

train.isna().sum().sort_values(ascending=False)


ENSG00000285476:AC139491.7    0
ENSG00000000003:TSPAN6        0
ENSG00000000005:TNMD          0
ENSG00000000419:DPM1          0
ENSG00000000457:SCYL3         0
                             ..
ENSG00000001617:SEMA3F        0
ENSG00000001626:CFTR          0
ENSG00000001629:ANKIB1        0
ENSG00000001630:CYP51A1       0
ENSG00000001631:KRIT1         0
Length: 17385, dtype: int64

In [70]:
from func_NT_cetux import *
n_hvg=1500
df=train
df_clean, dup = strip_prefix_from_genes(df, meta_cols=("cell_id","sample",'trattamento'), sep=":", on_duplicate="first")

adata_full = make_anndata_from_df(df_clean, set_raw=True)

In [71]:
import numpy as np
from scipy import sparse

X = adata_full.X

if sparse.issparse(X):
    n_nans = np.isnan(X.data).sum()
else:
    n_nans = np.isnan(X).sum()

print("Numero di NaN:", n_nans)

import numpy as np
from scipy import sparse


if sparse.issparse(X):
    X_dense = X.A  # converte in numpy array
else:
    X_dense = X

cells_with_nan = np.isnan(X_dense).any(axis=1)
genes_with_nan  = np.isnan(X_dense).any(axis=0)

print("Celle con NaN:", cells_with_nan.sum())
print("Geni con NaN :", genes_with_nan.sum())


Numero di NaN: 0
Celle con NaN: 0
Geni con NaN : 0


In [72]:
n_hvg = 1500
batch_key: str = DEFAULT_BATCH_KEY
def select_hvg_cell_ranger_(
    adata,
    n_top_genes=1000,
    batch_key=None,    
    subset=True, n_bins=50,
):
    sc.pp.highly_variable_genes(
        adata,
        flavor="seurat",
        n_top_genes=n_top_genes,
        batch_key=batch_key,  
        subset=subset
    )

select_hvg_cell_ranger_(adata_full, n_top_genes=n_hvg, batch_key=None, subset=True)

In [73]:
is_ref = (
    (adata_full.obs["sample"] == "CRC0322LMX") &
    (adata_full.obs["trattamento"] == "CTX1w_1")
)

adata_ref   = adata_full[is_ref].copy()

In [74]:
scale_and_pca(adata_ref, n_comps=42, max_value=10)
sc.pp.neighbors(adata_ref, n_pcs=20)
sc.tl.umap(adata_ref)

In [75]:
adata_full.obs["sample_tratt"] = (
    adata_full.obs["sample"].astype(str) + "_" +
    adata_full.obs["trattamento"].astype(str)
)

groups = adata_full.obs["sample_tratt"].unique()

In [76]:
ref_id='CRC0322LMX_CTX1w_1'
ingested_list = []

for g in groups:
    print(g)
    if g == ref_id:
        continue  # salta il reference
    
    print("→ Ingest di:", g)
    
    # Estraggo tutte le cellule di quella combinazione
    adata_q = adata_full[adata_full.obs["sample_tratt"] == g].copy()
    
    # Allineo i geni al reference (solo HVG del reference!)
    adata_q = adata_q[:, adata_ref.var_names].copy()
    
    # Ingest
    sc.tl.ingest(adata_q, adata_ref, embedding_method="umap")
    
    ingested_list.append(adata_q)

CRC0322LMO_CTX72h_1
→ Ingest di: CRC0322LMO_CTX72h_1


/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


CRC0322LMO_NT72h_1
→ Ingest di: CRC0322LMO_NT72h_1


/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


CRC0322LMX_CTX1w_1
CRC0322LMX_NT1w_1
→ Ingest di: CRC0322LMX_NT1w_1


/usr/local/mamba/envs/bbknn_env/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [77]:
adata_ingested = adata_ref.concatenate(*ingested_list)

/tmp/ipykernel_1199641/1901169737.py:1: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_ingested = adata_ref.concatenate(*ingested_list)


In [78]:
#sc.tl.ingest(adata_query, adata_ref, embedding_method="umap")
data_obs = pd.DataFrame(adata_ingested.obs)

#aggiungo umap alle obs perche di default sta nel obsm
data_obs["umap1"] = adata_ingested.obsm["X_umap"][:, 0]
data_obs["umap2"] = adata_ingested.obsm["X_umap"][:, 1]

In [79]:
path_to_save='/mnt/cold2/snaketree/prj/PPH/local/share/data/umap_ingested/'
data_path='/mnt/cold2/snaketree/prj/PPH/local/share/data/saver_mat'
for file in os.listdir(data_path):
    sample_name=str.split(file,sep='_')[4]
    trattamento=str.split(file,sep='_')[5]+'_'+str.split(file,sep='_')[6]
    trattamento_sample=sample_name+'_'+trattamento
    if 'merda' in sample_name:
        nome_file="_".join(str.split(file,sep='_')[4:7])+'_umap_ingested.csv'
        print('trovato lmo')
    else:
        nome_file="_".join(str.split(file,sep='_')[4:7])+'_singleron0722_umap_ingested.csv'
    if sample_name in train_id and trattamento in tratt_cercato:
        print(trattamento_sample)
        print(nome_file)
        tmp=data_obs[(data_obs['sample'] == sample_name) & (data_obs['trattamento'] == trattamento)]
        cols=['cell_id','umap1','umap2']
        print(tmp.head())
        tmp=tmp.loc[:,cols].reset_index(drop=True)
        file=path_to_save+nome_file
        tmp.to_csv(file)

CRC0322LMO_CTX72h_1
CRC0322LMO_CTX72h_1_singleron0722_umap_ingested.csv
                                                                     cell_id  \
uid                                                                            
CRC0322LMOCTX72h_1|AAACATCGAACCGAGAACGTATCA|CTX...  AAACATCGAACCGAGAACGTATCA   
CRC0322LMOCTX72h_1|AAACATCGAACGTGATGTACGCAA|CTX...  AAACATCGAACGTGATGTACGCAA   
CRC0322LMOCTX72h_1|AAACATCGAAGACGGACTAAGGTC|CTX...  AAACATCGAAGACGGACTAAGGTC   
CRC0322LMOCTX72h_1|AAACATCGAAGACGGAGAACAGGC|CTX...  AAACATCGAAGACGGAGAACAGGC   
CRC0322LMOCTX72h_1|AAACATCGAAGAGATCCTGGCATA|CTX...  AAACATCGAAGAGATCCTGGCATA   

                                                        sample trattamento  \
uid                                                                          
CRC0322LMOCTX72h_1|AAACATCGAACCGAGAACGTATCA|CTX...  CRC0322LMO    CTX72h_1   
CRC0322LMOCTX72h_1|AAACATCGAACGTGATGTACGCAA|CTX...  CRC0322LMO    CTX72h_1   
CRC0322LMOCTX72h_1|AAACATCGAAGACGGACTAAGGTC|CTX...  CRC